# Phase 8: Model Training & Hyperparameter Tuning

This notebook trains and compares multiple forecasting models for weekly Qty_Sold prediction.

## Objective
Train baseline and ML models, compare performance, tune hyperparameters, and save best models.

## Modeling Strategy
- **Time Series Approach**: Chronological split (train/val/test)
- **No leakage**: Future data never used for training
- **Demand-class aware**: Models selected based on product demand patterns
- **Grouped strategy**: Train one model per demand class to capture patterns

## Models Trained
1. **Baseline**: Naive, Moving Average
2. **Regression**: Linear Regression
3. **Tree-based**: Random Forest, Gradient Boosting, XGBoost

## Evaluation Metrics
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- MAPE (Mean Absolute Percentage Error)
- WAPE (Weighted Absolute Percentage Error)
- R²

## Input
- `data/processed/feature_ready_dataset.csv` (2600 rows × 29 features)

## Outputs
- `models/best_model_rf.pkl and models/best_model_gb.pkl` - Best models per demand class
- `data/outputs/model_metrics.xlsx` - Detailed metrics
- `data/outputs/best_models.csv` - Model selection summary
- `data/outputs/feature_importance.csv` - Feature importance ranking
- `reports/charts/model_comparison.png` - Performance comparison
- `reports/charts/feature_importance.png` - Feature importance chart
- `reports/charts/error_distribution.png` - Prediction error analysis

## 1. Setup and Data Loading

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

# Resolve project root whether executed from repo root or notebooks/ via nbconvert
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from models import (
    prepare_timeseries_data, train_baseline_models, train_ml_models,
    tune_random_forest, tune_gradient_boosting, save_model
)
from evaluate import evaluate_model, compare_models, get_feature_importance

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Imports complete')

Imports complete


In [2]:
# Create output directories
DATA_OUTPUTS = PROJECT_ROOT / 'data' / 'outputs'
DATA_OUTPUTS.mkdir(parents=True, exist_ok=True)

MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_CHARTS = PROJECT_ROOT / 'reports' / 'charts'
REPORTS_CHARTS.mkdir(parents=True, exist_ok=True)

# Load feature-ready dataset
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
df_all = pd.read_csv(DATA_PROCESSED / 'feature_ready_dataset.csv')

print(f'Data loaded: {df_all.shape}')
print(f'Demand classes: {df_all["Demand_Class"].unique()}')
print(f'Unique products: {df_all["Product_ID"].nunique()}')

Data loaded: (2600, 30)
Demand classes: <StringArray>
['Smooth']
Length: 1, dtype: str
Unique products: 25


## 2. Modeling Strategy

### Chronological Split
We use a chronological time series split:
- **Train**: Oldest 60% of data (weeks 1-62.4, oldest ~18 months)
- **Validation**: Middle 20% of data (weeks 62.4-83.2, recent ~6 months)
- **Test**: Latest 20% of data (weeks 83.2-104, most recent ~6 months)

### Grouped Modeling Strategy
Group by demand class and train separate models to capture class-specific patterns:
- **Smooth demand**: Regression-friendly, favor Linear/Tree models
- **Erratic demand**: Tree models excel with irregular patterns
- **Intermittent/Lumpy**: Baseline + robust ensemble models

In [3]:
# Sort by Week_End_Date only for chronological split
if 'Week_End_Date' not in df_all.columns:
    raise ValueError('Week_End_Date is required for chronological train/validation/test split')

df_all['Week_End_Date'] = pd.to_datetime(df_all['Week_End_Date'])
df_all = df_all.sort_values('Week_End_Date').reset_index(drop=True)

# Chronological split across the full timeline
def split_timeseries(df, train_pct=0.6, val_pct=0.2):
    n = len(df)
    train_end = int(n * train_pct)
    val_end = train_end + int(n * val_pct)
    
    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]
    
    return train, val, test

train_data, val_data, test_data = split_timeseries(df_all)

print(f'Train: {train_data.shape[0]} rows ({train_data["Week_End_Date"].min().date()} to {train_data["Week_End_Date"].max().date()})')
print(f'Val:   {val_data.shape[0]} rows ({val_data["Week_End_Date"].min().date()} to {val_data["Week_End_Date"].max().date()})')
print(f'Test:  {test_data.shape[0]} rows ({test_data["Week_End_Date"].min().date()} to {test_data["Week_End_Date"].max().date()})')
print(f'\nTotal: {train_data.shape[0] + val_data.shape[0] + test_data.shape[0]} rows')


Train: 1560 rows (2023-01-07 to 2024-03-16)


Val:   520 rows (2024-03-16 to 2024-08-10)
Test:  520 rows (2024-08-10 to 2024-12-28)

Total: 2600 rows


## 3. Model Training & Comparison

We train models on all data (grouped by demand class) to learn overall patterns.

In [4]:
# Prepare data for modeling
X_train, y_train = prepare_timeseries_data(train_data)
X_val, y_val = prepare_timeseries_data(val_data)
X_test, y_test = prepare_timeseries_data(test_data)

# Get feature names for importance analysis
feature_cols = [col for col in train_data.columns 
                if col not in ['Qty_Sold', 'Week_End_Date', 'Product_ID', 'Month_Start', 'Month_End', 'Season', 'Demand_Class']]

print(f'Features: {len(feature_cols)}')
print(f'Feature names: {feature_cols[:10]}... (showing first 10)')
print(f'\nTrain shape: {X_train.shape}')
print(f'Val shape: {X_val.shape}')
print(f'Test shape: {X_test.shape}')

Features: 23
Feature names: ['Year', 'Month', 'Quarter', 'ISO_Week', 'Week_Number', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_8']... (showing first 10)

Train shape: (1560, 23)
Val shape: (520, 23)
Test shape: (520, 23)


In [5]:
print('Training Baseline Models...')
baseline_results = train_baseline_models(X_train, y_train, X_val, y_val)

# Display results
baseline_metrics = {name: res['metrics'] for name, res in baseline_results.items()}
df_baseline = pd.DataFrame(baseline_metrics).T
print('\nBaseline Model Performance (Validation Set):')
print(df_baseline.round(4))

Training Baseline Models...

Baseline Model Performance (Validation Set):
                    MAE     RMSE     MAPE     WAPE      R2
Naive           40.3327  49.4668  60.1162  69.1836 -1.9731
Moving_Average  24.0385  30.0073  47.0975  41.2337 -0.0940


In [6]:
print('Training ML Models (without tuning)...')
ml_results = train_ml_models(X_train, y_train, X_val, y_val)

# Display results
ml_metrics = {name: res['metrics'] for name, res in ml_results.items()}
df_ml = pd.DataFrame(ml_metrics).T
print('\nML Model Performance (Validation Set):')
print(df_ml.round(4))

Training ML Models (without tuning)...



ML Model Performance (Validation Set):
                     MAE     RMSE     MAPE     WAPE      R2
LinearRegression  8.1943  11.7766  13.6959  14.0559  0.8315
RandomForest      9.2228  11.0418  16.0414  15.8201  0.8519
GradientBoosting  5.7682   7.4360   9.7122   9.8943  0.9328


## 4. Hyperparameter Tuning

Tune best-performing models using TimeSeriesSplit for proper time series validation.

In [7]:
print('Tuning Random Forest...')
best_rf, rf_tune_results = tune_random_forest(X_train, y_train, X_val, y_val, cv=3)

print(f'Best params: {rf_tune_results["best_params"]}')
print(f'\nTuned RF Metrics: {rf_tune_results["metrics"]}')

Tuning Random Forest...


Best params: {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 200}

Tuned RF Metrics: {'MAE': 9.381569986583125, 'RMSE': 10.952804234567846, 'MAPE': 16.848957020001652, 'WAPE': 16.09241759202779, 'R2': 0.85424154068}


In [8]:
print('Tuning Gradient Boosting...')
best_gb, gb_tune_results = tune_gradient_boosting(X_train, y_train, X_val, y_val, cv=3)

print(f'Best params: {gb_tune_results["best_params"]}')
print(f'\nTuned GB Metrics: {gb_tune_results["metrics"]}')

Tuning Gradient Boosting...


Best params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}

Tuned GB Metrics: {'MAE': 7.9058322207607965, 'RMSE': 8.733700870359389, 'MAPE': 18.20292455199134, 'WAPE': 13.561051475492706, 'R2': 0.9073213402162345}


## 5. Model Comparison

Compare all models and select best based on RMSE.

In [9]:
# Combine all model results
all_results = {
    **baseline_metrics,
    **ml_metrics,
    'RF_Tuned': rf_tune_results['metrics'],
    'GB_Tuned': gb_tune_results['metrics']
}

df_comparison = compare_models(all_results)
print('Model Comparison (sorted by RMSE):')
print(df_comparison.round(4))

# Save comparison
best_model_name = df_comparison.index[0]
print(f'\nBest Model: {best_model_name}')
print(f'Best RMSE: {df_comparison.iloc[0]["RMSE"]:.4f}')

Model Comparison (sorted by RMSE):
                      MAE     RMSE     MAPE     WAPE      R2
GradientBoosting   5.7682   7.4360   9.7122   9.8943  0.9328
GB_Tuned           7.9058   8.7337  18.2029  13.5611  0.9073
RF_Tuned           9.3816  10.9528  16.8490  16.0924  0.8542
RandomForest       9.2228  11.0418  16.0414  15.8201  0.8519
LinearRegression   8.1943  11.7766  13.6959  14.0559  0.8315
Moving_Average    24.0385  30.0073  47.0975  41.2337 -0.0940
Naive             40.3327  49.4668  60.1162  69.1836 -1.9731

Best Model: GradientBoosting
Best RMSE: 7.4360


## 6. Feature Importance Analysis

In [10]:
# Extract feature importance from tuned GB (best tree model)
df_importance = get_feature_importance(best_gb, feature_cols)

if df_importance is not None:
    print('Top 15 Important Features:')
    print(df_importance.head(15))
    
    # Normalize for visualization
    df_importance_viz = df_importance.copy()
    df_importance_viz['Importance'] = (df_importance_viz['Importance'] / 
                                       df_importance_viz['Importance'].sum())
else:
    print('Feature importance not available for selected model')

Top 15 Important Features:
            Feature  Importance
18  Inventory_Ratio    0.982294
0              Year    0.005792
10           Lag_12    0.004680
15    Rolling_Min_4    0.002628
12   Rolling_Mean_8    0.001938
14    Rolling_Max_4    0.000806
3          ISO_Week    0.000735
4       Week_Number    0.000598
11   Rolling_Mean_4    0.000303
22  Avg_Temperature    0.000165
1             Month    0.000044
9             Lag_8    0.000015
6             Lag_2    0.000000
2           Quarter    0.000000
5             Lag_1    0.000000


## 7. Save Results & Artifacts

In [11]:
# Save best models
save_model(best_rf, MODELS_DIR / 'best_model_rf.pkl')
save_model(best_gb, MODELS_DIR / 'best_model_gb.pkl')

print(f'Saved: {MODELS_DIR / "best_model_rf.pkl"}')
print(f'Saved: {MODELS_DIR / "best_model_gb.pkl"}')


Saved: D:\Final Project\models\best_model_rf.pkl
Saved: D:\Final Project\models\best_model_gb.pkl


In [12]:
# Save model comparison
excel_path = DATA_OUTPUTS / 'model_metrics.xlsx'
with pd.ExcelWriter(excel_path) as writer:
    df_comparison.to_excel(writer, sheet_name='Model_Comparison')
    train_data.head(10).to_excel(writer, sheet_name='Sample_Train', index=False)
    val_data.head(10).to_excel(writer, sheet_name='Sample_Val', index=False)

print(f'Saved: {excel_path}')

Saved: D:\Final Project\data\outputs\model_metrics.xlsx


In [13]:
# Save best model selection
best_models_info = pd.DataFrame([
    {
        'Model_Group': 'Grouped',
        'Selected_Model': best_model_name,
        'Model_File': 'best_model_gb.pkl',
        'MAE': df_comparison.loc[best_model_name, 'MAE'],
        'RMSE': df_comparison.loc[best_model_name, 'RMSE'],
        'MAPE': df_comparison.loc[best_model_name, 'MAPE'],
        'WAPE': df_comparison.loc[best_model_name, 'WAPE'],
        'R2': df_comparison.loc[best_model_name, 'R2']
    }
])

best_models_csv = DATA_OUTPUTS / 'best_models.csv'
best_models_info.to_csv(best_models_csv, index=False)
print(f'Saved: {best_models_csv}')
print(best_models_info)

Saved: D:\Final Project\data\outputs\best_models.csv
  Model_Group    Selected_Model         Model_File       MAE     RMSE  \
0     Grouped  GradientBoosting  best_model_gb.pkl  5.768183  7.43605   

       MAPE      WAPE        R2  
0  9.712202  9.894293  0.932816  


In [14]:
if df_importance is not None:
    feature_imp_path = DATA_OUTPUTS / 'feature_importance.csv'
    df_importance.to_csv(feature_imp_path, index=False)
    print(f'Saved: {feature_imp_path}')

Saved: D:\Final Project\data\outputs\feature_importance.csv


## 8. Visualizations

In [15]:
# Model comparison chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# RMSE
df_comparison['RMSE'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('RMSE by Model (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('RMSE')

# MAE
df_comparison['MAE'].plot(kind='barh', ax=axes[1], color='darkgreen')
axes[1].set_title('MAE by Model (Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('MAE')

# MAPE
df_comparison['MAPE'].plot(kind='barh', ax=axes[2], color='coral')
axes[2].set_title('MAPE by Model (Lower is Better)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('MAPE (%)')

plt.tight_layout()
chart_path = REPORTS_CHARTS / 'model_comparison.png'
fig.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Saved: {chart_path}')

Saved: D:\Final Project\reports\charts\model_comparison.png


In [16]:
if df_importance is not None and len(df_importance) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    top_n = min(15, len(df_importance))
    df_imp_top = df_importance.head(top_n).sort_values('Importance')
    
    ax.barh(df_imp_top['Feature'], df_imp_top['Importance'], color='steelblue')
    ax.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
    ax.set_title('Top 15 Feature Importance (Gradient Boosting)', fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    chart_path = REPORTS_CHARTS / 'feature_importance.png'
    fig.savefig(chart_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved: {chart_path}')

Saved: D:\Final Project\reports\charts\feature_importance.png


In [17]:
# Prediction errors on validation set
y_pred_best = gb_tune_results['predictions']
errors = y_val - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(errors, bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Prediction Error (Actual - Predicted)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Error Distribution (Validation Set)', fontsize=12, fontweight='bold')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].legend()

# Q-Q plot style
axes[1].scatter(y_val, y_pred_best, alpha=0.5, s=20)
min_val = min(y_val.min(), y_pred_best.min())
max_val = max(y_val.max(), y_pred_best.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Qty_Sold', fontsize=11)
axes[1].set_ylabel('Predicted Qty_Sold', fontsize=11)
axes[1].set_title('Actual vs Predicted (Validation Set)', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
chart_path = REPORTS_CHARTS / 'error_distribution.png'
fig.savefig(chart_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'Saved: {chart_path}')

Saved: D:\Final Project\reports\charts\error_distribution.png


## 9. Summary

In [18]:
print('\n' + '='*70)
print('PHASE 8 MODEL TRAINING COMPLETE'.center(70))
print('='*70)

print(f'\nData Split:')
print(f'  Train: {train_data.shape[0]:,} rows')
print(f'  Val:   {val_data.shape[0]:,} rows')
print(f'  Test:  {test_data.shape[0]:,} rows')

print(f'\nModels Trained: {len(all_results)}')
print(f'\nBest Model: {best_model_name}')
print(f'  MAE:  {df_comparison.loc[best_model_name, "MAE"]:.4f}')
print(f'  RMSE: {df_comparison.loc[best_model_name, "RMSE"]:.4f}')
print(f'  MAPE: {df_comparison.loc[best_model_name, "MAPE"]:.4f}%')
print(f'  WAPE: {df_comparison.loc[best_model_name, "WAPE"]:.4f}%')
print(f'  R2:   {df_comparison.loc[best_model_name, "R2"]:.4f}')

print(f'\nOutput Files:')
print(f'  Models: {MODELS_DIR}')
print(f'  Metrics: {DATA_OUTPUTS}')
print(f'  Charts: {REPORTS_CHARTS}')

print(f'\nPhase 8 Complete')
print('='*70)


                   PHASE 8 MODEL TRAINING COMPLETE                    

Data Split:
  Train: 1,560 rows
  Val:   520 rows
  Test:  520 rows

Models Trained: 7

Best Model: GradientBoosting
  MAE:  5.7682
  RMSE: 7.4360
  MAPE: 9.7122%
  WAPE: 9.8943%
  R2:   0.9328

Output Files:
  Models: D:\Final Project\models
  Metrics: D:\Final Project\data\outputs
  Charts: D:\Final Project\reports\charts

Phase 8 Complete


# Phase 8 Complete!

All models have been trained, evaluated, and saved. Artifacts and charts are available in the specified output folders.

- Models saved to: `models/`
- Metrics and summaries saved to: `data/outputs/`
- Charts saved to: `reports/charts/`

You can now proceed to the next phase or review the results and charts for insights.